In [1]:
# =========================
# CELL 1 — Setup & contracts
# =========================
from pathlib import Path
import sys, re
import numpy as np
import pandas as pd

ROOT = Path("/home/aidan/IMU_LM_Data")
sys.path.insert(0, str(ROOT))

from UTILS.helpers import (
    _canon, load_contracts, to_continuous_stream,
    run_qa_checks, check_sample_integrity,
    _split_on_gaps_seconds,
)

C          = load_contracts(ROOT)
SCHEMA     = C["SCHEMA"]
RAW2ID     = C["RAW2ID"]
ID2NAME    = C["ID2NAME"]
UNKNOWN_ID = C["UNKNOWN_ID"]
TARGET_HZ  = C["TARGET_HZ"]
CLEANED    = C["CLEANED"]

RAW_DIR      = ROOT / "data" / "raw_data" / "User_study_watch"
DATASET_NAME = "apple_watch_study"
GAP_CUTOFF_S = 0.5          # split into new segment if gap > 500 ms
G_MS2        = 9.80665      # standard gravity

# ---- Apple Watch CoreMotion 21-column layout (space-separated) ----
# 0        unix_ts              (s)
# 1-3      userAcceleration     (g, gravity-removed)
# 4-6      gravity              (normalised direction)
# 7-9      rotationRate         (rad/s)
# 10-12    magneticField        (µT)
# 13-15    attitude euler       (roll, pitch, yaw — rad)
# 16-19    attitude quaternion   (x, y, z, w)
# 20       elapsed_time         (s, monotonic clock)
WATCH_COLS = [
    "unix_ts",
    "user_acc_x", "user_acc_y", "user_acc_z",
    "gravity_x",  "gravity_y",  "gravity_z",
    "gyro_x",     "gyro_y",     "gyro_z",
    "mag_x",      "mag_y",      "mag_z",
    "euler_roll", "euler_pitch", "euler_yaw",
    "quat_x",    "quat_y",     "quat_z",    "quat_w",
    "elapsed_s",
]

# filename pattern: P{n}{activity}_{YY-MM-DD}_{HH-MM-SS}_motion(1).txt
# activity may contain letters AND digits (e.g. "setting3Dprinter")
FNAME_RE = re.compile(
    r"^(P\d+)([a-zA-Z][a-zA-Z0-9]*)_(\d{2}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})_motion(?:\(\d+\))?\.txt$"
)

# discover all motion files under subject sub-folders
motion_files = sorted(RAW_DIR.rglob("*_motion*.txt"))
# exclude old top-level files without activity labels
motion_files = [f for f in motion_files if FNAME_RE.match(f.name)]

print(f"RAW_DIR:    {RAW_DIR}")
print(f"TARGET_HZ:  {TARGET_HZ}")
print(f"Files:      {len(motion_files)}")
for f in motion_files:
    m = FNAME_RE.match(f.name)
    print(f"  • {f.name}  →  subject={m.group(1)}, activity={m.group(2)}")

RAW_DIR:    /home/aidan/IMU_LM_Data/data/raw_data/User_study_watch
TARGET_HZ:  50
Files:      70
  • P1airheart_26-03-10_17-58-41_motion(1).txt  →  subject=P1, activity=airheart
  • P1foldingpaper_26-03-10_17-53-37_motion(1).txt  →  subject=P1, activity=foldingpaper
  • P1handsanitizer_26-03-10_17-57-27_motion(1).txt  →  subject=P1, activity=handsanitizer
  • P1stircoffe_26-03-10_17-52-15_motion(1).txt  →  subject=P1, activity=stircoffe
  • P1unscrewbottle_26-03-10_18-02-30_motion(1).txt  →  subject=P1, activity=unscrewbottle
  • P1windingcord_26-03-10_18-00-34_motion(1).txt  →  subject=P1, activity=windingcord
  • P1writing_26-03-10_17-55-33_motion(1).txt  →  subject=P1, activity=writing
  • P10drawing_26-03-17_20-15-45_motion(1).txt  →  subject=P10, activity=drawing
  • P10fistpump_26-03-17_20-28-24_motion(1).txt  →  subject=P10, activity=fistpump
  • P10handgesturesequence_26-03-17_20-21-13_motion(1).txt  →  subject=P10, activity=handgesturesequence
  • P10rollingupposter_26-03-17_2

In [2]:
# =========================
# CELL 2 — Parse, resample to 50 Hz, build native frame
# =========================

def _resample_segment(seg_df, sensor_cols, target_hz=50):
    """Interpolate sensor columns onto a uniform 50 Hz grid."""
    t = seg_df["unix_ts"].to_numpy(np.float64)
    if t.size < 2:
        return pd.DataFrame()
    dt = 1.0 / target_hz
    grid = np.arange(t[0], t[-1] + 1e-12, dt, dtype=np.float64)
    if grid.size < 2:
        return pd.DataFrame()
    out = pd.DataFrame({"unix_ts": grid})
    for c in sensor_cols:
        y = seg_df[c].to_numpy(np.float64)
        m = np.isfinite(t) & np.isfinite(y)
        if m.sum() >= 2:
            out[c] = np.interp(grid, t[m], y[m]).astype(np.float32)
        else:
            out[c] = np.nan
    return out.dropna(subset=sensor_cols).reset_index(drop=True)


SENSOR_COLS = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]

# assign each unique activity label a numeric ID (1-based, alphabetical)
activity_names = sorted(set(
    FNAME_RE.match(f.name).group(2).lower()
    for f in motion_files if FNAME_RE.match(f.name)
))
ACTIVITY2ID = {name: np.int16(i + 1) for i, name in enumerate(activity_names)}
print("Activity → ID mapping:")
for a, i in ACTIVITY2ID.items():
    print(f"  {i}: {a}")

all_native = []

for fpath in motion_files:
    m = FNAME_RE.match(fpath.name)
    if not m:
        print(f"[SKIP] unrecognised filename: {fpath.name}")
        continue
    subject      = m.group(1)
    activity     = m.group(2).lower()
    session_tag  = m.group(3)

    raw = pd.read_csv(
        fpath, sep=r"\s+", header=None, names=WATCH_COLS,
        dtype=np.float64, on_bad_lines="skip",
    )
    if raw.empty or len(raw) < 2:
        print(f"[SKIP] empty/short: {fpath.name}")
        continue

    raw["acc_x"] = (raw["gravity_x"] + raw["user_acc_x"]) * G_MS2
    raw["acc_y"] = (raw["gravity_y"] + raw["user_acc_y"]) * G_MS2
    raw["acc_z"] = (raw["gravity_z"] + raw["user_acc_z"]) * G_MS2

    raw = raw.sort_values("unix_ts").dropna(
        subset=["unix_ts"] + SENSOR_COLS
    ).reset_index(drop=True)

    est_hz = 1.0 / np.median(np.diff(raw["unix_ts"].values))
    print(f"{fpath.name}  |  {len(raw):,} rows  |  Hz≈{est_hz:.1f}  |  {activity} (id={ACTIVITY2ID[activity]})")

    segments = _split_on_gaps_seconds(raw, "unix_ts", GAP_CUTOFF_S)
    for seg_i, seg in enumerate(segments):
        rs = _resample_segment(seg, SENSOR_COLS, TARGET_HZ)
        if rs.empty:
            continue
        rs["dataset"]                = DATASET_NAME
        rs["subject_id"]             = subject
        rs["session_id"]             = "1"
        rs["timestamp_ns"]           = (rs["unix_ts"] * 1e9).round().astype("int64")
        rs["dataset_activity_id"]    = ACTIVITY2ID[activity]
        rs["dataset_activity_label"] = activity
        all_native.append(rs)

if not all_native:
    raise SystemExit("No data produced — check RAW_DIR contents.")

native = pd.concat(all_native, ignore_index=True)[[
    "dataset", "subject_id", "session_id", "timestamp_ns",
    "acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z",
    "dataset_activity_id", "dataset_activity_label",
]].copy()

native["dataset"]                = native["dataset"].astype("string")
native["subject_id"]             = native["subject_id"].astype("string")
native["session_id"]             = native["session_id"].astype("string")
native["dataset_activity_label"] = native["dataset_activity_label"].astype("string")

print(f"\nNative frame: {len(native):,} rows  |  "
      f"{native['subject_id'].nunique()} subject(s)  |  "
      f"{native['session_id'].nunique()} session(s)")
print(f"\nActivity distribution:")
print(native[["dataset_activity_id", "dataset_activity_label"]].drop_duplicates().sort_values("dataset_activity_id").to_string(index=False))
native.head(3)

Activity → ID mapping:
  1: airheart
  2: applyhandstanitizer
  3: basketballshooting
  4: clickingpen
  5: crumblingpaper
  6: drawcircleinair
  7: drawfigureeight
  8: drawing
  9: drawingaircircle
  10: eatingwithchopsticks
  11: filingnails
  12: fistbump
  13: fistpump
  14: flickingbatmittonracket
  15: foldingpaper
  16: handgesturesequence
  17: handsanitizer
  18: karatechop
  19: massagegiving
  20: opencloselaptop
  21: opengatorade
  22: openingairpodscase
  23: organizefridge
  24: phonescrolling
  25: pointing
  26: pouringwaterbetweencups
  27: puttingonlotion
  28: readingjapanesemanga
  29: rockpaperscissors
  30: rollingupposter
  31: scratchingnails
  32: screwmasonjar
  33: setting3dprinter
  34: shadowboxing
  35: shuffledeckofcards
  36: snapping
  37: stircoffe
  38: stiringcoffe
  39: stiringcoffee
  40: takingphoneoutofpocket
  41: tearinguppaper
  42: tieshoelace
  43: turningbookpages
  44: twistingscrewdriver
  45: typing
  46: typingonkeybiard
  47: typingo

,dataset,subject_id,session_id,timestamp_ns,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,dataset_activity_id,dataset_activity_label
0,apple_watch_study,P1,1,1773183521922420992,0.313341,-5.267094,-8.531588,-0.070839,0.079437,-0.067855,1,airheart
1,apple_watch_study,P1,1,1773183521942420992,0.303679,-5.230183,-8.618328,-0.082216,0.096684,-0.078405,1,airheart
2,apple_watch_study,P1,1,1773183521962420992,0.294017,-5.193273,-8.705068,-0.093593,0.113930,-0.088955,1,airheart


In [3]:
# =========================
# CELL 3 — Schema conversion, QA, save
# =========================

df_final = to_continuous_stream(native, SCHEMA, RAW2ID, ID2NAME, UNKNOWN_ID)
print(f"Schema-aligned rows: {len(df_final):,}")
print(f"Columns: {list(df_final.columns)}\n")

run_qa_checks(df_final, SCHEMA, UNKNOWN_ID)
check_sample_integrity(df_final, SCHEMA)

# ---- Save ----
out_path = CLEANED / f"{DATASET_NAME}_clean_data.parquet"
out_path.parent.mkdir(parents=True, exist_ok=True)
df_final.to_parquet(out_path, index=False)
print(f"\n✓ Saved → {out_path}  ({out_path.stat().st_size / 1e6:.2f} MB)")

Schema-aligned rows: 364,551
Columns: ['dataset', 'subject_id', 'session_id', 'timestamp_ns', 'acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z', 'global_activity_id', 'global_activity_label', 'dataset_activity_id', 'dataset_activity_label']

Subjects: 10 | Sessions: 1
Monotonic violations (groups): 10
Median Hz: 50.00 (target=50)
Rows meeting required-not-null: 100.00%
Global mapping coverage: 3.6% (unknown=9000)

Top-15 dataset_activity_label:
dataset_activity_label
wavinggoodbye              14750
stiringcoffee              10927
writingwithpen             10027
pouringwaterbetweencups     9292
usingstyalus                8579
turningbookpages            8472
wrirtingonwhiteboard        7037
wipingglass                 6924
screwmasonjar               6918
takingphoneoutofpocket      6849
usingmouse                  6840
tieshoelace                 6819
zippingsleepingbag          6532
applyhandstanitizer         6382
eatingwithchopsticks        6316
Name: count, dtype: Int64

